# EFCAMDAT Preprocessing and Experimental Dataset Partitioning

## 1. Extract the Original EFCAMDAT XML to CSV

In [ ]:
import os
import xml.etree.ElementTree as ET
import csv
from google.colab import drive
import re

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================================================
# CONFIGURATION - ADJUST PATHS IF NEEDED
# =========================================================
# Replace with the exact path where the EFCAMDAT_Database.xml is stored in Drive
xml_input_path = "/content/drive/MyDrive/YOUR_PATH/EFCAMDAT_Database.xm"
csv_output_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_extracted_raw.csv"
# =========================================================

In [ ]:
if not os.path.exists(xml_input_path):
    raise FileNotFoundError(f"Could not find target file at: {xml_input_path}. Verify that the file name is exact.")

print(f"File confirmed. Initiating Attribute-Aware Data Pipeline...")

csv_headers = ["writing_id", "level", "unit", "learner_id", "nationality", "topic_id", "topic_title", "grade", "text"]

# Outer regex to extract the complete writing block pair cleanly
writing_block_regex = re.compile(r'<writing\s+([^>]+)>(.*?)</writing>', re.DOTALL)

# Highly precise inner regex maps matching the attribute structure of EFCAMDAT
attr_regexes = {
    "writing_id": re.compile(r'id=["\'](\d+)["\']'),
    "level": re.compile(r'level=["\'](\d+)["\']'),
    "unit": re.compile(r'unit=["\'](\d+)["\']'),
    "learner_id": re.compile(r'id=["\'](\d+)["\']'), # Parsed inside <learner> block
    "nationality": re.compile(r'nationality=["\']([a-zA-Z]+)["\']') # Parsed inside <learner> block
}

tag_regexes = {
    "learner_block": re.compile(r'<learner\s+([^>]+)/>'),
    "topic_block": re.compile(r'<topic\s+id=["\'](\d+)["\']>(.*?)</topic>'),
    "grade": re.compile(r'<grade>(\d+)</grade>'),
    "text": re.compile(r'<text>(.*?)</text>', re.DOTALL)
}

discovered_levels = set()
total_records = 0

with open(csv_output_path, mode='w', encoding='utf-8', newline='') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(csv_headers)

    chunk_size = 32 * 1024 * 1024 # 32MB chunks
    leftover = ""

    with open(xml_input_path, mode='r', encoding='utf-8', errors='ignore') as xml_file:
        while True:
            chunk = xml_file.read(chunk_size)
            if not chunk:
                break

            data_to_parse = leftover + chunk
            matches = writing_block_regex.findall(data_to_parse)

            for open_tag_attrs, block_content in matches:
                current_data = {h: "" for h in csv_headers}

                # 1. Parse top-level attributes from the writing opening tag string
                w_id_m = attr_regexes["writing_id"].search(open_tag_attrs)
                lvl_m = attr_regexes["level"].search(open_tag_attrs)
                unt_m = attr_regexes["unit"].search(open_tag_attrs)

                if w_id_m: current_data["writing_id"] = w_id_m.group(1)
                if lvl_m: current_data["level"] = lvl_m.group(1)
                if unt_m: current_data["unit"] = unt_m.group(1)

                # 2. Parse inline inner learner tag attributes
                learner_m = tag_regexes["learner_block"].search(block_content)
                if learner_m:
                    learner_attrs = learner_m.group(1)
                    l_id_m = attr_regexes["learner_id"].search(learner_attrs)
                    nat_m = attr_regexes["nationality"].search(learner_attrs)
                    if l_id_m: current_data["learner_id"] = l_id_m.group(1)
                    if nat_m: current_data["nationality"] = nat_m.group(1)

                # 3. Parse explicit tag elements
                topic_m = tag_regexes["topic_block"].search(block_content)
                if topic_m:
                    current_data["topic_id"] = topic_m.group(1)
                    current_data["topic_title"] = topic_m.group(2).strip()

                grade_m = tag_regexes["grade"].search(block_content)
                if grade_m:
                    current_data["grade"] = grade_m.group(1)

                text_m = tag_regexes["text"].search(block_content)
                if text_m:
                    raw_text = text_m.group(1).strip()
                    # Safe sanitize: remove stray nested tokens or escape components if any remain
                    raw_text = re.sub(r'<[^>]+>', '', raw_text)
                    raw_text = raw_text.replace("&quot;", '"').replace("&amp;", "&")
                    current_data["text"] = raw_text

                # Maintain data profile collection tracking
                if current_data["level"]:
                    lvl_num = int(current_data["level"])
                    discovered_levels.add(lvl_num)

                # Write record block to CSV framework
                if current_data["text"]:
                    writer.writerow([current_data[h] for h in csv_headers])
                    total_records += 1

                if total_records % 50000 == 0 and total_records > 0:
                    print(f" -> Successfully parsed & compiled {total_records:,} documents...")

            # Slide window securely
            last_idx = data_to_parse.rfind('</writing>')
            if last_idx != -1:
                leftover = data_to_parse[last_idx + 10:]
            else:
                leftover = data_to_parse

print("\n==================================================")
print(" 🎉 CRITICAL STRUCTURAL PARSING COMPLETE")
print("==================================================")
print(f"Total Records Saved to File: {total_records:,}")
print(f"Target Destination Path:     {csv_output_path}")
print("--------------------------------------------------")
print(" 🔍 COMPLETE SCALED INVENTORY LEVELS DISCOVERED:")
print(sorted(list(discovered_levels)))
print("==================================================")

File confirmed. Initiating Attribute-Aware Data Pipeline...
 -> Successfully parsed & compiled 50,000 documents...
 -> Successfully parsed & compiled 100,000 documents...
 -> Successfully parsed & compiled 150,000 documents...
 -> Successfully parsed & compiled 200,000 documents...
 -> Successfully parsed & compiled 250,000 documents...
 -> Successfully parsed & compiled 300,000 documents...
 -> Successfully parsed & compiled 350,000 documents...
 -> Successfully parsed & compiled 400,000 documents...
 -> Successfully parsed & compiled 450,000 documents...
 -> Successfully parsed & compiled 500,000 documents...
 -> Successfully parsed & compiled 550,000 documents...
 -> Successfully parsed & compiled 600,000 documents...
 -> Successfully parsed & compiled 650,000 documents...
 -> Successfully parsed & compiled 700,000 documents...
 -> Successfully parsed & compiled 750,000 documents...
 -> Successfully parsed & compiled 800,000 documents...
 -> Successfully parsed & compiled 850,000 do

## 2. Inspect the Original EFCAMDAT CEFR Distribution

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION - PATH TO YOUR NEW RAW CSV
# =========================================================
# Replace with the exact path where the efcamdat_extracted_raw.csv is stored in Drive
csv_input_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_extracted_raw.csv"
# =========================================================

print("Loading extracted CSV file into pandas (this might take a moment due to its scale)...")
# Reading only the 'level' column to keep execution incredibly fast and light on RAM
df = pd.read_csv(csv_input_path, usecols=["level"])

# 2. Define the official EF Curriculum to CEFR mapping framework
ef_to_cefr = {
    1: "A1", 2: "A1", 3: "A1",
    4: "A2", 5: "A2", 6: "A2",
    7: "B1", 8: "B1", 9: "B1",
    10: "B2", 11: "B2", 12: "B2",
    13: "C1", 14: "C1", 15: "C1",
    16: "C2"
}

print("Mapping numeric curriculum tiers to CEFR levels...")
# Clean up columns just in case any rows have stray text spacing
df['level'] = pd.to_numeric(df['level'], errors='coerce')
df['cefr'] = df['level'].map(ef_to_cefr)

# Drop rows that don't map to a valid level (if any exist)
df = df.dropna(subset=['cefr'])

# 3. Calculate and display distributions
total_valid_records = len(df)

print("\n==================================================")
print(" 📊 EFCAMDAT MASTER CEFR DISTRIBUTION")
print("==================================================")

cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

for level in cefr_order:
    count = (df['cefr'] == level).sum()
    percentage = (count / total_valid_records) * 100
    print(f"Level {level}: {count:>8,} samples | Allocation: {percentage:.2f}%")

print("--------------------------------------------------")
print(f"Total Combined CEFR Inventory Original Dataset: {total_valid_records:,}")
print("==================================================")

Loading extracted CSV file into pandas (this might take a moment due to its scale)...
Mapping numeric curriculum tiers to CEFR levels...

 📊 EFCAMDAT MASTER CEFR DISTRIBUTION
Level A1:  625,985 samples | Allocation: 53.04%
Level A2:  307,996 samples | Allocation: 26.09%
Level B1:  168,361 samples | Allocation: 14.26%
Level B2:   61,329 samples | Allocation: 5.20%
Level C1:   14,698 samples | Allocation: 1.25%
Level C2:    1,940 samples | Allocation: 0.16%
--------------------------------------------------
Total Combined CEFR Inventory Original Dataset: 1,180,309


## 3. Inspect the Cleaned EFCAMDAT Subcorpus Distribution

In [ ]:
# =========================================================
# CONFIGURATION - PATH TO YOUR NEW RAW CSV
# =========================================================
# Replace with the exact path where the ef_POStagged_original_corrected.csv is stored in Drive
csv_input_path = "/content/drive/MyDrive/YOUR_PATH/ef_POStagged_original_corrected.csv"
# =========================================================

print("Loading extracted CSV file into pandas (this might take a moment due to its scale)...")
# Reading only the 'level' column to keep execution incredibly fast and light on RAM
df = pd.read_csv(csv_input_path, usecols=["level"])

# 2. Define the official EF Curriculum to CEFR mapping framework
ef_to_cefr = {
    1: "A1", 2: "A1", 3: "A1",
    4: "A2", 5: "A2", 6: "A2",
    7: "B1", 8: "B1", 9: "B1",
    10: "B2", 11: "B2", 12: "B2",
    13: "C1", 14: "C1", 15: "C1",
    16: "C2"
}

print("Mapping numeric curriculum tiers to CEFR levels...")
# Clean up columns just in case any rows have stray text spacing
df['level'] = pd.to_numeric(df['level'], errors='coerce')
df['cefr'] = df['level'].map(ef_to_cefr)

# Drop rows that don't map to a valid level (if any exist)
df = df.dropna(subset=['cefr'])

# 3. Calculate and display distributions
total_valid_records = len(df)

print("\n==================================================")
print(" 📊 EFCAMDAT MASTER CEFR DISTRIBUTION")
print("==================================================")

cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

for level in cefr_order:
    count = (df['cefr'] == level).sum()
    percentage = (count / total_valid_records) * 100
    print(f"Level {level}: {count:>8,} samples | Allocation: {percentage:.2f}%")

print("--------------------------------------------------")
print(f"Total Combined CEFR Inventory Cleaned Dataset: {total_valid_records:,}")
print("==================================================")

Loading extracted CSV file into pandas (this might take a moment due to its scale)...
Mapping numeric curriculum tiers to CEFR levels...

 📊 EFCAMDAT MASTER CEFR DISTRIBUTION
Level A1:  312,298 samples | Allocation: 50.35%
Level A2:  170,196 samples | Allocation: 27.44%
Level B1:   96,165 samples | Allocation: 15.51%
Level B2:   34,151 samples | Allocation: 5.51%
Level C1:    7,396 samples | Allocation: 1.19%
Level C2:        0 samples | Allocation: 0.00%
--------------------------------------------------
Total Combined CEFR Inventory Cleaned Dataset: 620,206


## 4. Construct the EFCAMDAT Master Corpus


In [ ]:
import pandas as pd
import re
import os
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION - PATHS
# =========================================================
# replace with actula drive path
cleaned_csv_path = "/content/drive/MyDrive/YOUR_PATH/ef_POStagged_original_corrected.csv"

# replace with actula drive path
raw_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_extracted_raw.csv"

# replace with actula drive path
output_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined.csv"
# =========================================================

In [ ]:
print("\n2. Loading and extracting C2 samples from the Raw Dataset...")
df_raw = pd.read_csv(raw_csv_path, low_memory=False)

# Isolate Level 16 (C2)
df_raw['level'] = pd.to_numeric(df_raw['level'], errors='coerce')
df_c2 = df_raw[df_raw['level'] == 16].copy()

# Keep matching columns (ENSURING 'cefr' IS INCLUDED)
df_c2['cefr'] = "C2"
df_c2 = df_c2[['level', 'topic_id', 'topic_title', 'text', 'cefr']].copy()
df_c2.rename(columns={'text': 'clean_text'}, inplace=True)

print(" -> Applying Öksüz et al. cleaning standards to C2 samples...")
df_c2 = df_c2.dropna(subset=['clean_text'])
df_c2 = df_c2[df_c2['clean_text'].str.split().str.len() >= 20]

def sanitize_c2_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace("&quot;", '"').replace("&amp;", "&")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_c2['clean_text'] = df_c2['clean_text'].apply(sanitize_c2_text)
print(f" -> C2 extraction and cleaning complete: {len(df_c2):,} rows validated.")


print("\n3. Merging datasets...")
# Combine the Clean A1-C1 data with the newly sanitized C2 data
df_master = pd.concat([df_clean, df_c2], ignore_index=True)

# Safety catch: Re-map everything globally to fix any floating NaN values
df_master['cefr'] = df_master['level'].map(ef_to_cefr)

# Shuffle the dataset to ensure random distribution
df_master = df_master.sample(frac=1, random_state=42).reset_index(drop=True)

# =========================================================
# UPDATED: FINAL CALCULATED CEFR DISTRIBUTION REPORT
# =========================================================
print("\n==================================================")
print(" 📊 EFCAMDAT MASTER CEFR DISTRIBUTION")
print("==================================================")

total_combined_records = len(df_master)
cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

for level in cefr_order:
    count = (df_master['cefr'] == level).sum()
    percentage = (count / total_combined_records) * 100 if total_combined_records > 0 else 0.0
    print(f"Level {level}: {count:>8,} samples | Allocation: {percentage:.2f}%")

print("--------------------------------------------------")
print(f"Total Combined CEFR Inventory Master Dataset: {total_combined_records:,}")
print("==================================================")


print("\n4. Saving to Google Drive...")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_master.to_csv(output_path, index=False)
print(f"✅ Success! Unified master dataset saved to: {output_path}")


2. Loading and extracting C2 samples from the Raw Dataset...
 -> Applying Öksüz et al. cleaning standards to C2 samples...
 -> C2 extraction and cleaning complete: 1,916 rows validated.

3. Merging datasets...

 📊 EFCAMDAT MASTER CEFR DISTRIBUTION
Level A1:  312,298 samples | Allocation: 50.20%
Level A2:  170,196 samples | Allocation: 27.36%
Level B1:   96,165 samples | Allocation: 15.46%
Level B2:   34,151 samples | Allocation: 5.49%
Level C1:    7,396 samples | Allocation: 1.19%
Level C2:    1,916 samples | Allocation: 0.31%
--------------------------------------------------
Total Combined CEFR Inventory Master Dataset: 622,122

4. Saving to Google Drive...
✅ Success! Unified master dataset saved to: /content/drive/MyDrive/Masters Thesis/dataset/EFCAMDAT Corpus/PROCESSED/efcamdat_master_combined.csv


## 5. Normalize HTML Formatting Artifacts

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION - PATHS
# =========================================================
# replace with actula drive path
master_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined.csv"
# =========================================================

In [ ]:
print("\nLoading master dataset...")
df = pd.read_csv(master_csv_path, low_memory=False)

print("Processing and stripping HTML line break tags...")

# Ensure the column is treated as strings to avoid attribute errors on missing values
df['clean_text'] = df['clean_text'].astype(str)

# Global string replacements covering common variations of HTML line breaks
df['clean_text'] = df['clean_text'].str.replace("<br/>", " ", regex=False)
df['clean_text'] = df['clean_text'].str.replace("<br />", " ", regex=False)
df['clean_text'] = df['clean_text'].str.replace("<br>", " ", regex=False)

# Clean up any duplicated white spaces resulting from the tag removal
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Saving changes back to Google Drive...")
df.to_csv(master_csv_path, index=False)

print(f"✅ Success! All line breaks removed. File saved to: {master_csv_path}")

## 6. Validate C2 Error-Annotation Cleaning

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# =========================================================
# CONFIGURATION & AUTHENTICATION
# =========================================================
hf_token = "HF_TOKEN"
login(token=hf_token)

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
# Keeping your repo_id stored here for the future hidden-state extraction phase
repo_id = "MohammadKhosravi/cefr-llama3.1-8b-hidden-states-combined"

print("\n1. Loading Tokenizer and Model (Optimized for Colab VRAM limits)...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Loaded in float16 to keep VRAM footprint under ~15GB for T4 compatibility
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token
)
model.eval()


1. Loading Tokenizer and Model (Optimized for Colab VRAM limits)...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [ ]:
# =========================================================
# REFINED SYSTEM PROMPT (FEW-SHOT)
# =========================================================
system_prompt = """You are an elite linguistic editor specializing in cleaning up language learner corpora (EFCAMDAT).
Your objective is to read texts containing inline teacher error annotations and output a perfectly smooth, grammatically flawless English text.

CRITICAL TRANSFORMATION RULES:
1. Handle Fused Annotations (Pattern: [WrongWord][UPPERCASE_CODE][CorrectedWord]):
   ALWAYS extract and keep ONLY the [CorrectedWord].
   - 'gardernSPgarden' -> 'garden'
   - 'appliesWCuseful' -> 'useful'
   - 'isDPRin' -> 'in' (delete 'is', keep 'in')
   - 'Get AGGetsArrestedA' -> 'Gets Arrested'
2. Handle Stray Tags: Delete lone uppercase error codes (e.g., PU, RS, MW, VT, D).
Below is the official EFCAMDAT Error Code dictionary used to tag edits:

- XC: change from x to y
- AG: agreement
- AR: article
- AS: add space
- CO: combine sentences
- C: capitalization
- D: delete
- EX: expression of idiom
- HL: highlight
- IS: insert
- MW: missing word
- NS: new sentence
- NSW: no such word
- PH: phraseology
- PL: plural
- PO: possessive
- PR: preposition
- PS: part of speech
- PU: punctuation
- RS: remove space
- SI: singular
- SP: spelling
- VT: verb tense
- WC: word choice
- WO: word order

3. Fix Hidden Errors: Fix obvious typos not marked by teachers (e.g., 'is was' -> 'it was', 'every people' -> 'everyone').
4. Do NOT over-edit: Do not change valid words (e.g., do not change 'home' to 'house').

OUTPUT FORMAT:
You must reply with ONLY the cleaned text. No preambles, no explanations, no "Step 1", and no markdown formatting.

EXAMPLE INPUT:
I am walking towards the lake through a colorful gardernSPgarden. The stones feelsAGfeel fresh.

EXAMPLE OUTPUT:
I am walking towards the lake through a colorful garden. The stones feel fresh."""

In [ ]:
# =========================================================
# THE 5 USER TEST SAMPLES
# =========================================================
test_samples = {
    "Sample 1 (Fused Codes Mixed)": (
        "The annual international AI conference took place last weekend. Among all the robots there, three has particularly "
        "caught my attention. The first was a Japanese service robot, that is being improved to become a companion robot. "
        "It is already able to show some signs of emotional intelligence, and although it still don't look like a human yet, "
        "it can make some face expressions and even burp. The second was aARan African service robot called Florence. It is "
        "already in use collecting the trash and helping with the security of a hospital. In the near futurePU, Florence will "
        "also be programmed PRto visit patients in quarantine and will be taller, as she is just two feet tall at the moment. "
        "The third was developed by anARa retired Major, whichWCwho had plenty of contact with service and combat robots during "
        "his career, but decided to create a leisure robot. His robot was a sport robot and he is even planingWCplanning to "
        "organise competitions involving only this kind of machines. RS"
    ),
    "Sample 2 (Heavy Structural Fusions)": (
        "The three robots discussed in the conference are appliesWCuseful isDPRin veryWCmany different fields. It is my opinion "
        "that the second type of robots which is the one with ARthe hospital helpingWCduty quality will be ARthe most beneficial "
        "to the well-being of humanPLhumans. ThisWCThese robotPLrobots can replace humanPLhumans in the jobs that are time-consuming "
        "like watching surveillance tapePLtapes, or risky MWduties like visiting a DquarantineWCquarantined patientPLpatients. "
        "While the other two types of robots are more to the point thatWCand MWcan entertainsAGentertain people. These two types "
        "of robots are goodsWCgood for society that already meets theirEXits citizens basic needs, as the countries that they Dare "
        "developing in whichEXthem isAGare Japan and ARthe USA. Last of all, I still strongly believe that the development of "
        "robots is to help but not to replace the human connection."
    ),
    "Sample 3 (Clean / Unmarked Grammatical Check)": (
        "Christ the Redeemer is a statue of Jesus Christ in Rio de Janeiro, Brazil. It is considered the largest Art Deco "
        "statue in the world and the 5th largest statue of Jesus in the world. It is located at the peak of the 700-metre "
        "Corcovado mountain in the Tijuca Forest National Park overlooking the city of Rio de Janeiro. The Statue is a symbol "
        "of Brazilian Christianity, and has become an icon for Rio de Janeiro and Brazil. The statue is made of reinforced "
        "concrete and soapstone, and was constructed between 1922 and 1931. The statue of Christ the Redeemer with open arms "
        "symbolizes peace and welcomes every people in the city of Rio. It also means that Brazil, as a very catholic country, "
        "recognizes the deity of Jesus Christ. On July 7, 2007, Christ the Redeemer was named one of the New Seven Wonders "
        "of the World, in Lisbon."
    ),
    "Sample 4 (Complex Typo Extraction)": (
        "In my work it is very importantPU, the writingMW part, as a Civil Engineer, I have to do many kinds of worksSPwork, "
        "for example;PU, I have many years, writing a list of activities to do every day, I write in a daily reminder all kind "
        "of writings, but not only write in my daily book but I have to send letters, memorandums, e-mails and worksSPwork minutes. "
        "I also have to write many kind of documents like technical specification of the projects, legal documents for biddings, "
        "fill up a construction logsSIlog, description of functioning procedures and redaction of scientific studies, as you "
        "can see, I am very involved in the document redaction. weCWe know that theD redaction is like a finger print of our "
        "character and personality, that's the reason why some timesSPsometimes it is possible to know whoMW has writeVTwritten "
        "something, according with the redaction structure, used expressions or simply grammar, so redaction itD is one of "
        "the most important skill as a professional person."
    ),
    "Sample 5 (Preposition Fusions & Hidden Bug Case)": (
        "I am sitting on a chair atPRon my balcony. A delectable and exquisite breakfast is in front of me on the table. "
        "The sun is raising and the first gentle shafts of sunlight are trying to warm up my body. I can hear the birds "
        "singing and the leaves rustling. This quite morning is soothing and calming me. I gaze toPRat the enchanting scenery. "
        "The leaves of the trees have already this special red yellow color of the summer. The wet morning dew and dump are "
        "slowly vanishing.The soft breeze and smell of my fresh coffee is wafting towards me. I take ARa mouthful of this "
        "incredible tasty coffee. I feel how the warm drink goes through my gullet and gets into my stomach. My body gets "
        "warm and soothing. Nothing can upset me now, as it is a perfect morning. I am in a serene mood and this led me "
        "to drift slowly, gently, towards sleep.\n\n"
        "--- Hidden Bug Test Case Added Here ---\n"
        "My husband and I are lucky enough to own our own home. We have good jobs and a solid credit history. When my father "
        "died, he left me some money and we used that as a down payment on a home loan. I feel very fortunate. Looking back "
        "is was far from easy, but owning our own home makes all the trouble well worth it."
    )
}

In [ ]:
# =========================================================
# INFERENCE ENGINE LOOP
# =========================================================
print("\n2. Executing LLM Editorial Pipeline over test samples...\n")

for name, raw_text in test_samples.items():
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Clean this text:\n\n{raw_text}"}
    ]

    # Render chat structures cleanly
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=len(inputs.input_ids[0]) + 150, # Generous overhead for text rewrites
            temperature=0.1,  # Low temperature forces high adherence to constraints
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Strip generation prefix out
    cleaned_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    print("=" * 70)
    print(f"📝 INPUT SOURCED: {name}")
    print("=" * 70)
    print(raw_text)
    print("-" * 70)
    print("✨ LLM CLEANED GRAMMATICAL OUTPUT:")
    print("-" * 70)
    print(cleaned_output)
    print("=" * 70 + "\n")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



2. Executing LLM Editorial Pipeline over test samples...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


📝 INPUT SOURCED: Sample 1 (Fused Codes Mixed)
The annual international AI conference took place last weekend. Among all the robots there, three has particularly caught my attention. The first was a Japanese service robot, that is being improved to become a companion robot. It is already able to show some signs of emotional intelligence, and although it still don't look like a human yet, it can make some face expressions and even burp. The second was aARan African service robot called Florence. It is already in use collecting the trash and helping with the security of a hospital. In the near futurePU, Florence will also be programmed PRto visit patients in quarantine and will be taller, as she is just two feet tall at the moment. The third was developed by anARa retired Major, whichWCwho had plenty of contact with service and combat robots during his career, but decided to create a leisure robot. His robot was a sport robot and he is even planingWCplanning to organise competitions invol

## 7. Clean Residual Error Annotations in C2 Texts

Appendix: Error codes: <br/>
Code Meaning <br/>
XC change from x to y <br/>
AG agreement <br/>
AR article <br/>
AS add space <br/>
CO combine sentences <br/>
C capitalization <br/>
D delete <br/>
EX expression of idiom <br/>
HL highlight <br/>
IS insert <br/>
MW missing word <br/>
NS new sentence <br/>
NSW no such word <br/>
PH phraseology <br/>
PL plural <br/>
PO possessive <br/>
PR preposition <br/>
PS part of speech <br/>
PU punctuation <br/>
RS remove space <br/>
SI singular <br/>
SP spelling <br/>
VT verb tense <br/>
WC word choice <br/>
WO word order <br/>

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm
import math
import re
import os
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# 1. CONFIGURATION & AUTHENTICATION
# =========================================================
hf_token = "HF_TOKEN"
login(token=hf_token)

model_id = "meta-llama/Llama-3.1-8B-Instruct"

# replace with actula drive path
input_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined.csv"
output_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined_C2_Cleaned.csv"

# A100 Optimization: 32 is a very safe and extremely fast batch size for an 80GB GPU.
BATCH_SIZE = 32


In [ ]:
# =========================================================
# 2. LOAD DATASET & SPLIT
# =========================================================
print(f"Loading master dataset from: {input_csv_path}")
df = pd.read_csv(input_csv_path, low_memory=False)

# Isolate C2 samples and untouched A1-C1 samples
df_c2 = df[df['cefr'] == 'C2'].copy()
df_non_c2 = df[df['cefr'] != 'C2'].copy()

print(f"Found {len(df_c2)} C2 rows for processing, and {len(df_non_c2)} non-C2 rows to preserve.")


Loading master dataset from: /content/drive/MyDrive/Mohammd_Thesis/efcamdat_master_combined.csv
Found 1916 C2 rows for processing, and 620206 non-C2 rows to preserve.


In [ ]:
# =========================================================
# 3. PREPARE OPTIMIZED MODEL & TOKENIZER FOR BATCHING
# =========================================================
print("\nLoading Tokenizer and Model in bfloat16 for A100...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
# CRITICAL FOR BATCHING: Decoder-only models must use left-padding
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Optimal precision for A100 Ampere architecture
    device_map="auto",
    token=hf_token
)
model.eval()


Loading Tokenizer and Model in bfloat16 for A100...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [ ]:
# =========================================================
# 4. PROMPT ENGINEERING
# =========================================================
system_prompt = """You are an elite linguistic editor specializing in cleaning up language learner corpora (EFCAMDAT).
Your objective is to read texts containing inline teacher error annotations and output a perfectly smooth, grammatically flawless English text.

CRITICAL TRANSFORMATION RULES:
1. Handle Fused Annotations (Pattern: [WrongWord][UPPERCASE_CODE][CorrectedWord]):
   ALWAYS extract and keep ONLY the [CorrectedWord].
   - 'gardernSPgarden' -> 'garden'
   - 'appliesWCuseful' -> 'useful'
   - 'isDPRin' -> 'in' (delete 'is', keep 'in')
   - 'Get AGGetsArrestedA' -> 'Gets Arrested'
2. Handle Stray Tags: Delete lone uppercase error codes (e.g., PU, RS, MW, VT, D).
Below is the official EFCAMDAT Error Code dictionary used to tag edits:

- XC: change from x to y
- AG: agreement
- AR: article
- AS: add space
- CO: combine sentences
- C: capitalization
- D: delete
- EX: expression of idiom
- HL: highlight
- IS: insert
- MW: missing word
- NS: new sentence
- NSW: no such word
- PH: phraseology
- PL: plural
- PO: possessive
- PR: preposition
- PS: part of speech
- PU: punctuation
- RS: remove space
- SI: singular
- SP: spelling
- VT: verb tense
- WC: word choice
- WO: word order

3. Fix Hidden Errors: Fix obvious typos not marked by teachers (e.g., 'is was' -> 'it was', 'every people' -> 'everyone').
4. Do NOT over-edit: Do not change valid words (e.g., do not change 'home' to 'house').

OUTPUT FORMAT:
You must reply with ONLY the cleaned text. No preambles, no explanations, no "Step 1", and no markdown formatting.

EXAMPLE INPUT:
I am walking towards the lake through a colorful gardernSPgarden. The stones feelsAGfeel fresh.

EXAMPLE OUTPUT:
I am walking towards the lake through a colorful garden. The stones feel fresh."""

# Convert all C2 texts into chat-formatted prompt strings
formatted_prompts = []
for text in df_c2['clean_text'].tolist():
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Clean this text:\n\n{text}"}
    ]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    formatted_prompts.append(prompt_str)

In [ ]:
# =========================================================
# 5. BATCHED INFERENCE PIPELINE
# =========================================================
print(f"\n🚀 Starting High-Speed Batched Inference on A100 (Batch Size: {BATCH_SIZE})...")

cleaned_texts = []
num_batches = math.ceil(len(formatted_prompts) / BATCH_SIZE)

# Process in chunks
for i in tqdm(range(0, len(formatted_prompts), BATCH_SIZE), total=num_batches, desc="Batches Processed"):
    batch_prompts = formatted_prompts[i : i + BATCH_SIZE]

    # Tokenize the batch and push to GPU
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=False).to("cuda")

    # Generate the text
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,  # Generous buffer for longer essays
            temperature=0.1,     # Highly deterministic
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Slice the output to get ONLY the new generated tokens, stripping the prompt context out
    generation_only = outputs[:, inputs.input_ids.shape[1]:]

    # Decode the batch and strip whitespace/special tokens
    batch_decoded = tokenizer.batch_decode(generation_only, skip_special_tokens=True)
    cleaned_texts.extend([text.strip() for text in batch_decoded])


🚀 Starting High-Speed Batched Inference on A100 (Batch Size: 32)...


Batches Processed:   0%|          | 0/60 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Batches Processed: 100%|██████████| 60/60 [30:57<00:00, 30.95s/it]


In [ ]:
# =========================================================
# 6. RECOMBINE AND SAVE TO NEW FILE
# =========================================================
print("\nRecombining cleaned C2 text with the rest of the dataset...")
# Update only the clean_text column for C2
df_c2['clean_text'] = cleaned_texts

# Concat back with the preserved A1-C1 data
df_master_final = pd.concat([df_non_c2, df_c2], ignore_index=True)

# Reshuffle dataset for general hygiene
df_master_final = df_master_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Saving newly compiled dataset to safe location...")
df_master_final.to_csv(output_csv_path, index=False)

print(f"\n✅ SUCCESS! Master dataset protected. Cleaned output saved to:\n{output_csv_path}")

## 8. Validate and Finalize the EFCAMDAT Master Corpus

In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION
# =========================================================
# replace with actula drive path
master_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined_C2_Cleaned.csv"
# =========================================================

print("\nLoading master dataset for Diagnostic Health Check...\n")
df = pd.read_csv(master_csv_path, low_memory=False)

# Keep track of whether the dataset passes all checks
passed_all_checks = True




Loading master dataset for Diagnostic Health Check...



In [ ]:
print("==================================================")
print(" 🩺 DATASET HEALTH DIAGNOSTIC REPORT")
print("==================================================")
# ---------------------------------------------------------
# CHECK 1: Missing Values
# ---------------------------------------------------------
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    print("❌ FAILED: Missing Values Detected!")
    print(missing_counts[missing_counts > 0])
    passed_all_checks = False
else:
    print("✅ PASSED: No missing values in any column.")

 🩺 DATASET HEALTH DIAGNOSTIC REPORT
✅ PASSED: No missing values in any column.


In [ ]:
# ---------------------------------------------------------
# CHECK 2: Numeric Integer Types (level, topic_id)
# ---------------------------------------------------------
numeric_cols = ['level', 'topic_id']
numeric_issues = 0
for col in numeric_cols:
    # Check if there are any non-numeric or float values holding decimals
    non_ints = df[~df[col].apply(lambda x: str(x).isdigit())]
    if len(non_ints) > 0:
        print(f"❌ FAILED: Column '{col}' contains {len(non_ints)} non-integer values.")
        numeric_issues += 1

if numeric_issues == 0:
    print("✅ PASSED: 'level' and 'topic_id' contain strictly integer values.")
else:
    passed_all_checks = False

✅ PASSED: 'level' and 'topic_id' contain strictly integer values.


In [ ]:
# ---------------------------------------------------------
# CHECK 3: String Types (topic_title, clean_text)
# ---------------------------------------------------------
string_cols = ['topic_title', 'clean_text']
string_issues = 0
for col in string_cols:
    # Check if every single cell is fundamentally a Python string
    non_strs = df[df[col].apply(type) != str]
    if len(non_strs) > 0:
        print(f"❌ FAILED: Column '{col}' contains {len(non_strs)} non-string values (e.g., floats/NaNs).")
        string_issues += 1

if string_issues == 0:
    print("✅ PASSED: 'topic_title' and 'clean_text' are strictly strings.")
else:
    passed_all_checks = False

✅ PASSED: 'topic_title' and 'clean_text' are strictly strings.


In [ ]:
# ---------------------------------------------------------
# CHECK 4: CEFR Format Validation
# ---------------------------------------------------------
valid_cefr = {'A1', 'A2', 'B1', 'B2', 'C1', 'C2'}
invalid_cefr = df[~df['cefr'].isin(valid_cefr)]

if len(invalid_cefr) > 0:
    print(f"❌ FAILED: 'cefr' column contains {len(invalid_cefr)} invalid formats.")
    print(f"   Unique invalid values found: {invalid_cefr['cefr'].unique()}")
    passed_all_checks = False
else:
    print("✅ PASSED: 'cefr' column formatting is perfectly valid (A1-C2).")


✅ PASSED: 'cefr' column formatting is perfectly valid (A1-C2).


In [ ]:
# ---------------------------------------------------------
# CHECK 5: Residual Error Codes & HTML Tags Detection (DEEP SCAN WITH CEFR STATS)
# ---------------------------------------------------------
import re

suspicious_codes = ['XC', 'HL', 'MW', 'NS', 'NSW', 'PU', 'RS', 'VT', 'SP', 'WC']
codes_regex = '|'.join(suspicious_codes)

# Regex looks for: HTML tags OR standalone suspicious codes OR codes fused between lowercase letters
anomaly_pattern = re.compile(rf'<[^>]+>|\b({codes_regex})\b|[a-z]+({codes_regex})[a-z]+')

print("Scanning text for residual fusions and HTML tags (Deep CEFR Profile)...")
# Safely find row boolean mask
anomaly_mask = df['clean_text'].astype(str).str.contains(anomaly_pattern, regex=True, na=False)
anomalies_df = df[anomaly_mask]

if len(anomalies_df) > 0:
    print(f"\n⚠️ WARNING: Found {len(anomalies_df):,} total anomaly matches across the dataset.")
    print("--------------------------------------------------")
    print(" 📊 ANOMALY DISTRIBUTION BY CEFR LEVEL")
    print("--------------------------------------------------")

    cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

    for level in cefr_order:
        # Calculate how many anomalies are in this specific tier
        level_anomalies = len(anomalies_df[anomalies_df['cefr'] == level])
        # Calculate the total rows owned by this tier in the master dataframe
        level_total = len(df[df['cefr'] == level])

        # Determine proportional scale impact
        percentage_of_level = (level_anomalies / level_total * 100) if level_total > 0 else 0.0
        percentage_of_total_anomalies = (level_anomalies / len(anomalies_df) * 100)

        print(f"Level {level}: {level_anomalies:>5,} anomalies | "
              f"Affects {percentage_of_level:.2f}% of this class | "
              f"Holds {percentage_of_total_anomalies:.2f}% of all anomalies")

    print("--------------------------------------------------")

    print("\n   --- Dynamic Sample Assessment ---")
    # Take a diverse slice of samples across levels to visualize structural anomalies
    sample_rows = anomalies_df.head(5)
    for idx, row in sample_rows.iterrows():
        text_str = str(row['clean_text'])
        match_obj = anomaly_pattern.search(text_str)
        trigger = match_obj.group() if match_obj else "Unknown"
        print(f"   [{row['cefr']}] Row {idx:<6} Trigger [{trigger}]: {text_str[:120]}...")
else:
    print("✅ PASSED: No suspicious error codes or HTML tags detected.")

print("==================================================")

Scanning text for residual fusions and HTML tags (Deep CEFR Profile)...


/tmp/ipykernel_2433/2184054310.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  anomaly_mask = df['clean_text'].astype(str).str.contains(anomaly_pattern, regex=True, na=False)



⚠️ WARNING: Found 1,749 total anomaly matches across the dataset.
--------------------------------------------------
 📊 ANOMALY DISTRIBUTION BY CEFR LEVEL
--------------------------------------------------
Level A1: 1,019 anomalies | Affects 0.33% of this class | Holds 58.26% of all anomalies
Level A2:   458 anomalies | Affects 0.27% of this class | Holds 26.19% of all anomalies
Level B1:   197 anomalies | Affects 0.20% of this class | Holds 11.26% of all anomalies
Level B2:    52 anomalies | Affects 0.15% of this class | Holds 2.97% of all anomalies
Level C1:    17 anomalies | Affects 0.23% of this class | Holds 0.97% of all anomalies
Level C2:     6 anomalies | Affects 0.31% of this class | Holds 0.34% of all anomalies
--------------------------------------------------

   --- Dynamic Sample Assessment ---
   [B1] Row 420    Trigger [<change>]: I thought about the food and drinks, The meat was overcooked. <change><selection>The meat's taste, and smells looked lik...
   [B2] Row 531 

In [ ]:
# =========================================================
# PURGE ANOMALIES & RE-VERIFY PIPELINE
# =========================================================
print("\n🔥 Initiating absolute purge of anomaly matches...")
original_row_count = len(df)

# Use the inverse of our boolean mask (~) to retain ONLY clean rows
df = df[~anomaly_mask].copy()

purged_count = original_row_count - len(df)
print(f" -> Successfully deleted {purged_count:,} flagged records.")
print(f" -> Current secure dataset volume: {len(df):,} rows.")

print("\n🔍 Executing secondary verification sweep...")
# Re-run the regex scan over the newly filtered dataframe
post_purge_mask = df['clean_text'].astype(str).str.contains(anomaly_pattern, regex=True, na=False)
remaining_anomalies = post_purge_mask.sum()

if remaining_anomalies == 0:
    print("✅ VERIFIED: Secondary scan completed with exactly 0 structural anomalies remaining!")
    passed_all_checks = True  # Flips the final status monitor flag to True
else:
    print(f"❌ FAILED: Secondary sweep detected {remaining_anomalies} persistent noise fragments.")
    passed_all_checks = False

print("==================================================")


🔥 Initiating absolute purge of anomaly matches...
 -> Successfully deleted 1,749 flagged records.
 -> Current secure dataset volume: 620,373 rows.

🔍 Executing secondary verification sweep...


/tmp/ipykernel_2433/3790066943.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  post_purge_mask = df['clean_text'].astype(str).str.contains(anomaly_pattern, regex=True, na=False)


✅ VERIFIED: Secondary scan completed with exactly 0 structural anomalies remaining!


In [ ]:
# =========================================================
# FINAL DISTRIBUTION QUALITY VERIFICATION
# =========================================================
print("\n==================================================")
print(" 📊 POST-PURGE FINAL CEFR LEVEL DISTRIBUTION")
print("==================================================")

total_final_records = len(df)
cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2"]

for level in cefr_order:
    count = (df['cefr'] == level).sum()
    percentage = (count / total_final_records) * 100 if total_final_records > 0 else 0.0
    print(f"Level {level}: {count:>8,} samples | Allocation: {percentage:.2f}%")

print("--------------------------------------------------")
print(f"Total Secured Inventory Master Dataset: {total_final_records:,}")
print("==================================================")


 📊 POST-PURGE FINAL CEFR LEVEL DISTRIBUTION
Level A1:  311,279 samples | Allocation: 50.18%
Level A2:  169,738 samples | Allocation: 27.36%
Level B1:   95,968 samples | Allocation: 15.47%
Level B2:   34,099 samples | Allocation: 5.50%
Level C1:    7,379 samples | Allocation: 1.19%
Level C2:    1,910 samples | Allocation: 0.31%
--------------------------------------------------
Total Secured Inventory Master Dataset: 620,373


In [ ]:
# ---------------------------------------------------------
# ADDING A UNIQUE ID
# ---------------------------------------------------------
if 'text_id' not in df.columns:
    print("\nGenerating unique 'text_id' for dataset tracking...")
    # Creates IDs formatted like: EFCAM_000001, EFCAM_000002...
    df.insert(0, 'text_id', [f"EFCAM_{i:06d}" for i in range(1, len(df) + 1)])

    # Save the dataset back with the new IDs
    df.to_csv(master_csv_path, index=False)
    print(f"✅ Unique IDs appended and saved to {master_csv_path}.")
else:
    print("\n✅ Dataset already contains a 'text_id' column.")

print("==================================================")
if passed_all_checks:
    print("🎉 STATUS: ALL STRUCTURAL CHECKS PASSED. Ready for subsets!")
else:
    print("🛑 STATUS: DATASET REQUIRES FIXES BEFORE SPLITTING.")
print("==================================================")


Generating unique 'text_id' for dataset tracking...
✅ Unique IDs appended and saved to /content/drive/MyDrive/Mohammd_Thesis/efcamdat_master_combined_C2_Cleaned.csv.
🎉 STATUS: ALL STRUCTURAL CHECKS PASSED. Ready for subsets!


## 9. Construct the Steering Training Dataset and Evaluation-Judge Dataset

In [ ]:
import pandas as pd
import os
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION - I/O PATHS
# =========================================================
# replace with actula drive path
master_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined_C2_Cleaned.csv"

# Output Paths for the new subsets
classifier_output_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_classifier_subset.csv"
judge_output_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_eval_judge_subset.csv"


In [ ]:
# =========================================================
# TARGET ALLOCATIONS (EFCAMDAT CONTRIBUTIONS)
# =========================================================
# The exact number of samples we need to pull from EFCAMDAT for the Classifier
classifier_targets = {
    "A1": 14655,
    "A2": 12889,
    "B1": 10850,
    "B2": 10570,
    "C1": 2765,
    "C2": 928
}

# The target number of samples for the Eval Judge
# (We use 25000 for the lower tiers, but gracefully accept 'whatever is left' if we hit the ceiling)
judge_targets = {
    "A1": 25000,
    "A2": 25000,
    "B1": 25000,
    "B2": 25000,
    "C1": 4614, # ~ 7379 - 2765
    "C2": 982   # ~ 1910 - 928
}

In [ ]:
# =========================================================
# PROCESSING PIPELINE
# =========================================================
print(f"\nLoading and shuffling master dataset...")
df = pd.read_csv(master_csv_path, low_memory=False)

# SHUFFLE ENTIRE DATASET to guarantee maximum randomness in selection
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop unnecessary columns
columns_to_keep = ['text_id', 'level', 'cefr', 'clean_text']
df = df[columns_to_keep]

classifier_rows = []
judge_rows = []

print("Executing strict mathematical data split...")

for cefr_level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
    # Isolate all rows for the current level
    df_level = df[df['cefr'] == cefr_level]

    c_target = classifier_targets[cefr_level]
    j_target = judge_targets[cefr_level]

    # 1. Slice for Classifier
    df_classifier_slice = df_level.iloc[:c_target]
    classifier_rows.append(df_classifier_slice)

    # 2. Slice for Eval Judge (Take from exactly where the Classifier stopped)
    # Using min() ensures we don't try to pull more rows than actually exist (e.g., for B2)
    end_idx = min(c_target + j_target, len(df_level))
    df_judge_slice = df_level.iloc[c_target:end_idx]
    judge_rows.append(df_judge_slice)

# Recombine the slices into final DataFrames
df_classifier = pd.concat(classifier_rows, ignore_index=True)
df_judge = pd.concat(judge_rows, ignore_index=True)

# Final reshuffle for good measure before saving
df_classifier = df_classifier.sample(frac=1, random_state=101).reset_index(drop=True)
df_judge = df_judge.sample(frac=1, random_state=202).reset_index(drop=True)



Loading and shuffling master dataset...
Executing strict mathematical data split...


In [ ]:
# =========================================================
# REPORTING
# =========================================================
print("\n==================================================")
print(" 📊 CLASSIFIER HEAD SUBSET (EFCAMDAT CONTRIBUTION)")
print("==================================================")
for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
    count = (df_classifier['cefr'] == level).sum()
    print(f"Level {level}: {count:>8,} samples")
print("--------------------------------------------------")
print(f"Total Classifier Subset: {len(df_classifier):,}")
print("*(Ready to be merged with HF universal dataset)*")
print("==================================================")

print("\n==================================================")
print(" ⚖️ EVAL JUDGE SUBSET (PURE EFCAMDAT)")
print("==================================================")
for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
    count = (df_judge['cefr'] == level).sum()
    print(f"Level {level}: {count:>8,} samples")
print("--------------------------------------------------")
print(f"Total Eval Judge Subset: {len(df_judge):,}")
print("==================================================")



 📊 CLASSIFIER HEAD SUBSET (EFCAMDAT CONTRIBUTION)
Level A1:   14,655 samples
Level A2:   12,889 samples
Level B1:   10,850 samples
Level B2:   10,570 samples
Level C1:    2,765 samples
Level C2:      928 samples
--------------------------------------------------
Total Classifier Subset: 52,657
*(Ready to be merged with HF universal dataset)*

 ⚖️ EVAL JUDGE SUBSET (PURE EFCAMDAT)
Level A1:   25,000 samples
Level A2:   25,000 samples
Level B1:   25,000 samples
Level B2:   23,529 samples
Level C1:    4,614 samples
Level C2:      982 samples
--------------------------------------------------
Total Eval Judge Subset: 104,125


In [ ]:
# =========================================================
# SAVE TO DISK
# =========================================================
print("\nSaving subsets to Google Drive...")
df_classifier.to_csv(classifier_output_path, index=False)
df_judge.to_csv(judge_output_path, index=False)
print("✅ Success! Both datasets seamlessly partitioned and saved.")


Saving subsets to Google Drive...
✅ Success! Both datasets seamlessly partitioned and saved.


In [ ]:
# =========================================================
# DATA LEAKAGE VERIFICATION SWEEP
# =========================================================
print("🕵️ Running cross-dataset leakage validation...")

# Extract unique text identifier manifolds as sets
classifier_ids = set(df_classifier['text_id'])
judge_ids = set(df_judge['text_id'])

# Calculate the intersection (overlap) between the two pools
shared_ids = classifier_ids.intersection(judge_ids)

print("\n==================================================")
print(" 🛑 DATA LEAKAGE ANALYSIS REPORT")
print("==================================================")
print(f"Unique IDs in Classifier Subset: {len(classifier_ids):,}")
print(f"Unique IDs in Eval Judge Subset: {len(judge_ids):,}")
print("--------------------------------------------------")
print(f"Number of Overlapping/Shared IDs: {len(shared_ids)}")
print("==================================================")

if len(shared_ids) == 0:
    print("🎉 STATUS: 100% LEAK-PROOF! Zero matching IDs detected.")
    print("Both datasets are mathematically isolated for secure training.")
else:
    print("🛑 CRITICAL WARNING: Data leakage detected between subsets!")
    print(f"Please inspect the following leaked IDs: {list(shared_ids)[:5]}...")
print("==================================================")

🕵️ Running cross-dataset leakage validation...

 🛑 DATA LEAKAGE ANALYSIS REPORT
Unique IDs in Classifier Subset: 52,657
Unique IDs in Eval Judge Subset: 104,125
--------------------------------------------------
Number of Overlapping/Shared IDs: 0
🎉 STATUS: 100% LEAK-PROOF! Zero matching IDs detected.
Both datasets are mathematically isolated for secure training.


## 10 - Creating The Benchmark Prompt Subset

In [ ]:
import pandas as pd
import itertools
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [ ]:
# =========================================================
# CONFIGURATION - I/O PATHS
# =========================================================
# replace with actula drive path
master_csv_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_master_combined_C2_Cleaned.csv"
benchmark_output_path = "/content/drive/MyDrive/YOUR_PATH/efcamdat_benchmark_prompts_subset.csv"


In [ ]:
# =========================================================
# PIPELINE EXECUTION
# =========================================================
print(f"\nLoading master dataset...")
df = pd.read_csv(master_csv_path, low_memory=False)

# 1. Extract only unique topic pairs
print("Isolating unique topic pairs...")
unique_topics = df[['topic_id', 'topic_title']].drop_duplicates().dropna()
unique_topics['topic_id'] = unique_topics['topic_id'].astype(int)

num_unique_topics = len(unique_topics)
print(f" -> Found exactly {num_unique_topics} unique topics in the master dataset.")

# 2. Define the 6 target CEFR execution levels
cefr_levels = ["A1", "A2", "B1", "B2", "C1", "C2"]

# 3. Create the crossed factorial design matrix (128 topics * 6 levels)
print(f"Generating balanced factorial grid ({num_unique_topics} topics x 6 CEFR levels)...")
grid_rows = []

for _, row in unique_topics.iterrows():
    for level in cefr_levels:
        grid_rows.append({
            "topic_id": row["topic_id"],
            "topic_title": row["topic_title"],
            "cefr": level
        })

# Convert list of dicts back into a structured DataFrame
df_benchmark = pd.DataFrame(grid_rows)

# Sort logically by topic_id and CEFR tier for beautiful readability
df_benchmark = df_benchmark.sort_values(by=["topic_id", "cefr"]).reset_index(drop=True)



Loading master dataset...
Isolating unique topic pairs...
 -> Found exactly 128 unique topics in the master dataset.
Generating balanced factorial grid (128 topics x 6 CEFR levels)...


In [ ]:
# =========================================================
# REPORTING & VERIFICATION
# =========================================================
print("\n==================================================")
# Check balance verification matrix
print(" 📊 BENCHMARK PROMPT DISTRIBUTION TRACKER")
print("==================================================")
for level in cefr_levels:
    count = (df_benchmark['cefr'] == level).sum()
    print(f"Target Level {level}: {count:>4} prompt configurations")
print("--------------------------------------------------")
print(f"Total Evaluation Matrix Combinations: {len(df_benchmark)}")
print("==================================================")



 📊 BENCHMARK PROMPT DISTRIBUTION TRACKER
Target Level A1:  128 prompt configurations
Target Level A2:  128 prompt configurations
Target Level B1:  128 prompt configurations
Target Level B2:  128 prompt configurations
Target Level C1:  128 prompt configurations
Target Level C2:  128 prompt configurations
--------------------------------------------------
Total Evaluation Matrix Combinations: 768


> **Note:** The initial prompt matrix contained 768 conditions (128 prompts × 6 CEFR levels). During manual inspection, 11 prompts were found to elicit unintended behavior, such as role-play or conversational responses (e.g., adopting a shop-assistant role), rather than the intended direct text-generation task. These prompts and their corresponding six CEFR conditions were therefore removed (11 × 6 = 66 conditions). The final In-Domain Evaluation Prompt Matrix consequently contains 117 prompts × 6 CEFR levels = **702 conditions**, which is the dataset used for the experiments and reported in the thesis.

In [ ]:
# =========================================================
# SAVE TO DRIVE
# =========================================================
print(f"Saving pristine benchmark grid to Drive...")
df_benchmark.to_csv(benchmark_output_path, index=False)
print(f"✅ Success! Benchmark prompts file saved to:\n{benchmark_output_path}")